# Candidate Screening — RAG + FAISS on top of the QLoRA Adapter

This notebook is the third stage of the pipeline: it starts from the
LoRA adapter saved at the end of `candidate_screening_qlora_qwen.ipynb`
(the QLoRA fine-tune of `Qwen/Qwen2.5-3B-Instruct` as a regression
scorer) and adds a FAISS retrieval layer on top of it.

**What "RAG" means for this specific model.** The fine-tuned model is a
regression head (`AutoModelForSequenceClassification`, `num_labels=1`),
not a text generator — the LoRA adapter and the `score` head were
trained to output a single float, not to write text. So retrieval here
does **not** mean "retrieve context, then have an LLM generate a score."
It means:

1. **Calibration/ensemble** — blend the regression head's score with a
   similarity-weighted average of nearby training candidates' scores.
2. **Out-of-distribution flagging** — warn when a candidate has no close
   match in the training data, i.e. the regression head is extrapolating.
3. **Explainability** — retrieve the candidates that most resemble a new
   one and use them as grounding for a natural-language rationale
   (generated by a *separate* model call, not the regression head).

**Section 0 rebuilds `train_df` / `val_df` / `test_df` from scratch**,
using the identical loading, `input_text` construction, and stratified
split code from notebook 1, with the same `SEED`. Because the split is
deterministic, this reproduces the exact same three dataframes without
requiring you to have saved them separately — as long as you point
`DATA_PATH` at the same source file used in notebook 1.

**What to attach on Kaggle (right sidebar → "+ Add Input"):**
- The **adapter** saved by notebook 1 (`qwen-qlora-screening-score-adapter/`)
  — upload it as a Kaggle Dataset (or Model) and attach it here.
- The **same raw data file** (`candidates_extracted.json`) used in notebook 1.

```
pip install -q -U transformers peft accelerate datasets scikit-learn scipy \
    "bitsandbytes>=0.46.1" torchao faiss-gpu
```
Restart the runtime after this install, before running anything else.
`faiss-gpu` needs a CUDA runtime that matches the notebook's GPU — if the
install fails on your Kaggle GPU image, use `faiss-cpu` instead; the
index sizes here are small enough that CPU FAISS is plenty fast.


In [1]:
!pip install -q faiss-cpu
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.4 MB/s eta 0:00:00


In [2]:
import json
import re
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import faiss

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Config — update these three paths to match your attached inputs

In [3]:
RAW_DATA_INPUT_DIR = "/kaggle/input/datasets/divyeshbhavsar/qwen-qlora-screening-score-adapter"          # same dataset as notebook 1
DATA_PATH = f"/kaggle/input/datasets/divyeshbhavsar/qwen-qlora-screening-score-adapter/candidate_extracted (2).json"

ADAPTER_INPUT_DIR = "/kaggle/input/datasets/divyeshbhavsar/qwen-qlora-screening-score-adapter"  # <-- update

OUTPUT_DIR = "/kaggle/working"
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_LENGTH = 256


## 0. Rebuild `train_df` / `val_df` / `test_df` — identical to notebook 1

Same loading, `input_text` construction, title bucketing, and stratified
70/15/15 split, with the same `SEED`. This has to match notebook 1
exactly, or the FAISS index and the test evaluation below stop being
comparable to the numbers you already have.

In [4]:
with open(DATA_PATH, "r") as f:
    records = json.load(f)

df = pd.DataFrame(records)

if "id" not in df.columns:
    df["id"] = df.index

score_col = next((c for c in df.columns if c.strip().lower().replace(" ", "_") == "screening_score"), None)
if score_col is None:
    raise KeyError(f"No column matching 'screening_score' found. Columns: {df.columns.tolist()}")
if score_col != "screening_score":
    df = df.rename(columns={score_col: "screening_score"})

df["screening_score"] = pd.to_numeric(df["screening_score"], errors="coerce")
df = df.dropna(subset=["screening_score"]).reset_index(drop=True)

print(f"Loaded {len(df)} rows with a valid screening_score")


Loaded 1266 rows with a valid screening_score


In [5]:
def build_input_text(row) -> str:
    parts = []
    for field, label in [
        ("job_title", "Job Title"), ("experience", "Experience"),
        ("education", "Education"), ("other_info", "Other Info"),
        ("location", "Location"),
    ]:
        value = row.get(field)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{label}: {value}")
    return " | ".join(parts)


df["input_text"] = df.apply(build_input_text, axis=1)


In [6]:
def get_primary_title(job_title) -> str:
    if pd.isna(job_title) or not str(job_title).strip():
        return "Unknown"
    first = re.split(r"[,/]", str(job_title))[0]
    return first.strip().lower()


df["primary_title"] = df["job_title"].apply(get_primary_title)

MIN_SAMPLES_PER_CLASS = 10
title_counts = df["primary_title"].value_counts()
rare_titles = title_counts[title_counts < MIN_SAMPLES_PER_CLASS].index
df["strat_title"] = df["primary_title"].where(~df["primary_title"].isin(rare_titles), "Other")

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

train_df, temp_df = train_test_split(
    df, test_size=(VAL_FRAC + TEST_FRAC), stratify=df["strat_title"], random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
    stratify=temp_df["strat_title"], random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")


train: 886  val: 190  test: 190


## 1. Load the quantized base model and attach the LoRA adapter

Same `BitsAndBytesConfig` as notebook 1 — the adapter was trained against
this exact quantization, so it needs to be loaded the same way for the
weights to line up. `PeftModel.from_pretrained` wraps the quantized base
model with the trained adapter and the trained `score` head.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_INPUT_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=1,
    problem_type="regression",
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base_model, ADAPTER_INPUT_DIR)
model.eval()
print("Adapter loaded.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adapter loaded.


## 2. One forward pass, two outputs: regression score + pooled embedding

Qwen2's sequence-classification head pools the **last non-padded
token's** hidden state (not a `[CLS]` token like BERT) before the
`score` head sees it. Pulling `hidden_states[-1]` from the same call
that produces `logits` gives you the embedding for free — no second
model, no second forward pass.

In [8]:
@torch.no_grad()
def score_and_embed(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    outputs = model(**inputs, output_hidden_states=True)

    regression_score = outputs.logits.item()

    last_hidden = outputs.hidden_states[-1]                      # (1, seq_len, hidden_dim)
    seq_len = inputs["attention_mask"].sum(dim=1) - 1
    pooled_embedding = last_hidden[0, seq_len[0], :].float().cpu().numpy()

    return regression_score, pooled_embedding


@torch.no_grad()
def embed_batch(texts, batch_size: int = 16):
    """Same pooling as score_and_embed, batched for building the index."""
    all_scores, all_embeddings = [], []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start:start + batch_size]
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                            max_length=MAX_LENGTH, padding=True).to(model.device)
        outputs = model(**inputs, output_hidden_states=True)

        last_hidden = outputs.hidden_states[-1]
        seq_lens = inputs["attention_mask"].sum(dim=1) - 1
        for i in range(len(chunk)):
            all_embeddings.append(last_hidden[i, seq_lens[i], :].float().cpu().numpy())
        all_scores.extend(outputs.logits.squeeze(-1).float().cpu().tolist())

    return np.asarray(all_scores), np.stack(all_embeddings)


## 3. Build the FAISS index — from `train_df` only

`val_df` and `test_df` are queries later, never index members — same
held-out discipline as notebook 1's leakage checks. Embeddings are
L2-normalized so inner product in `IndexFlatIP` is cosine similarity.

In [9]:
_, train_embeddings = embed_batch(train_df["input_text"].tolist())
train_embeddings = train_embeddings.astype("float32")
faiss.normalize_L2(train_embeddings)

train_scores = train_df["screening_score"].to_numpy()
train_ids = train_df["id"].to_numpy()
train_titles = train_df["job_title"].fillna("Unknown").to_numpy()

index = faiss.IndexFlatIP(train_embeddings.shape[1])
index.add(train_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dim={train_embeddings.shape[1]}")


FAISS index built: 886 vectors, dim=2048


In [10]:
# Persist the index and its metadata so future notebooks/inference jobs
# don't have to rebuild it from scratch.
faiss.write_index(index, f"{OUTPUT_DIR}/candidate_faiss.index")

metadata_df = pd.DataFrame({
    "row": np.arange(len(train_ids)),
    "id": train_ids,
    "job_title": train_titles,
    "screening_score": train_scores,
})
metadata_df.to_csv(f"{OUTPUT_DIR}/candidate_faiss_metadata.csv", index=False)
print(f"Saved index and metadata to {OUTPUT_DIR}")


Saved index and metadata to /kaggle/working


## 4. The retrieval + blend function

`k` neighbors are retrieved, similarity-weighted-averaged into a
`knn_score`, then blended with the regression score by `alpha`
(`alpha=1.0` is pure regression, `alpha=0.0` is pure kNN).
`min_similarity` drives the out-of-distribution flag.

In [11]:
def score_with_rag(text: str, alpha: float = 0.7, k: int = 5, min_similarity: float = 0.5):
    regression_score, embedding = score_and_embed(text)

    query = embedding.reshape(1, -1).astype("float32")
    faiss.normalize_L2(query)
    similarities, neighbor_idx = index.search(query, k)
    similarities, neighbor_idx = similarities[0], neighbor_idx[0]

    neighbor_scores = train_scores[neighbor_idx]
    knn_score = float(np.average(neighbor_scores, weights=similarities))

    final_score = alpha * regression_score + (1 - alpha) * knn_score
    low_confidence = bool(similarities[0] < min_similarity)

    neighbors = [
        {"id": train_ids[i], "job_title": train_titles[i],
         "screening_score": float(train_scores[i]), "similarity": float(sim)}
        for sim, i in zip(similarities, neighbor_idx)
    ]

    return {
        "regression_score": regression_score,
        "knn_score": knn_score,
        "final_score": final_score,
        "low_confidence": low_confidence,
        "top_similarity": float(similarities[0]),
        "neighbors": neighbors,
    }


## 5. Tune `alpha` on `val_df` — not on `test_df`

Sweep candidate blend weights and pick whichever minimizes MAE on the
validation set. `test_df` stays untouched until the next section, same
as notebook 1's Section 11.

In [12]:
ALPHA_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
K = 5

val_texts = val_df["input_text"].tolist()
val_labels = val_df["screening_score"].to_numpy()

# Score once per candidate (regression + embedding + neighbors), reuse across alphas.
val_results = [score_with_rag(t, alpha=1.0, k=K) for t in val_texts]  # alpha=1.0 just to get regression+knn+neighbors
val_regression = np.array([r["regression_score"] for r in val_results])
val_knn = np.array([r["knn_score"] for r in val_results])

print("alpha | val MAE")
best_alpha, best_mae = None, np.inf
for alpha in ALPHA_GRID:
    blended = alpha * val_regression + (1 - alpha) * val_knn
    mae = mean_absolute_error(val_labels, blended)
    print(f"{alpha:5.2f} | {mae:.3f}")
    if mae < best_mae:
        best_alpha, best_mae = alpha, mae

print(f"\nBest alpha on val: {best_alpha} (val MAE {best_mae:.3f})")


alpha | val MAE
 0.50 | 18.094
 0.60 | 17.936
 0.70 | 17.853
 0.80 | 17.853
 0.90 | 17.889
 1.00 | 17.985

Best alpha on val: 0.8 (val MAE 17.853)


## 6. Final test evaluation — locked `alpha`, run once

Compares the RAG-blended score against the regression-only score on the
same `test_df`, same metrics as notebook 1's Section 11 (MAE, RMSE,
Spearman), so the two are directly comparable.

In [13]:
test_texts = test_df["input_text"].tolist()
test_labels = test_df["screening_score"].to_numpy()

test_results = [score_with_rag(t, alpha=best_alpha, k=K) for t in test_texts]
test_regression = np.array([r["regression_score"] for r in test_results])
test_final = np.array([r["final_score"] for r in test_results])

def report(name, preds):
    mae = mean_absolute_error(test_labels, preds)
    rmse = np.sqrt(mean_squared_error(test_labels, preds))
    spearman = spearmanr(preds, test_labels).correlation
    print(f"{name:22s}  MAE={mae:.3f}  RMSE={rmse:.3f}  Spearman={spearman:.3f}")
    return mae, rmse, spearman

print(f"(alpha = {best_alpha}, k = {K})\n")
report("Regression only", test_regression)
report("RAG-blended", test_final)

n_low_confidence = sum(r["low_confidence"] for r in test_results)
print(f"\n{n_low_confidence}/{len(test_results)} test candidates flagged low-confidence "
      f"(top neighbor similarity below 0.5)")


(alpha = 0.8, k = 5)

Regression only         MAE=15.946  RMSE=24.669  Spearman=0.568
RAG-blended             MAE=16.033  RMSE=23.877  Spearman=0.577

1/190 test candidates flagged low-confidence (top neighbor similarity below 0.5)


## 7. Explainability — retrieval-grounded rationale (optional, separate model)

The regression head only produces a number. This cell builds the prompt
you'd send to a *generative* model (a plain, non-quantized-for-regression
Qwen2.5-3B-Instruct, or whichever LLM your keyword-extraction stage
already uses) to turn a score plus its nearest neighbors into a
sentence a recruiter can actually read. Loading a second model is
optional and commented out — the prompt-building itself has no extra
dependency.

In [14]:
import json

def get_full_candidate_info(candidate_id, df):
    """Look up the full extracted-JSON row for a given id."""
    match = df[df["id"] == candidate_id]
    if match.empty:
        return None
    return match.iloc[0].to_dict()


def inspect_retrieval(candidate_id_or_row, result, source_df, train_df, fields=None):
    """
    Show full extracted info for the target candidate and each retrieved
    neighbor, so you can eyeball WHY they matched.

    candidate_id_or_row: either an id to look up in source_df, or a dict
                          (raw row) if the candidate isn't in any df (e.g. brand new).
    result: the dict returned by score_with_rag(...)
    source_df: the df the target came from (test_df, val_df, or None for new candidates)
    train_df: always train_df — neighbors are always indexed from train
    fields: which columns to show; defaults to the useful ones
    """
    if fields is None:
        fields = ["job_title", "experience", "education", "other_info", "location", "screening_score"]

    print("=" * 70)
    print("TARGET CANDIDATE")
    print("=" * 70)
    if isinstance(candidate_id_or_row, dict):
        target_info = candidate_id_or_row
    else:
        target_info = get_full_candidate_info(candidate_id_or_row, source_df)
    print(json.dumps({k: target_info.get(k) for k in fields if k in target_info}, indent=2, default=str))
    print(f"\npredicted_score (final): {result['final_score']:.1f}")

    print("\n" + "=" * 70)
    print("NEIGHBORS (from train_df)")
    print("=" * 70)
    for n in result["neighbors"]:
        full = get_full_candidate_info(n["id"], train_df)
        print(f"\n--- neighbor id={n['id']}  similarity={n['similarity']:.2f} ---")
        print(json.dumps({k: full.get(k) for k in fields if k in full}, indent=2, default=str))

In [15]:
target_id = test_df.iloc[15]["id"]
result = score_with_rag(test_df.iloc[15]["input_text"], alpha=best_alpha, k=K)
inspect_retrieval(target_id, result, test_df, train_df)

TARGET CANDIDATE
{
  "job_title": "data scientist",
  "experience": null,
  "education": null,
  "other_info": null,
  "location": "United States",
  "screening_score": 30
}

predicted_score (final): 32.8

NEIGHBORS (from train_df)

--- neighbor id=982  similarity=1.00 ---
{
  "job_title": "data scientist",
  "experience": null,
  "education": null,
  "other_info": null,
  "location": "United States",
  "screening_score": 35
}

--- neighbor id=1065  similarity=1.00 ---
{
  "job_title": "data scientist",
  "experience": null,
  "education": null,
  "other_info": null,
  "location": "United States",
  "screening_score": 30
}

--- neighbor id=962  similarity=1.00 ---
{
  "job_title": "data scientist",
  "experience": null,
  "education": null,
  "other_info": null,
  "location": "United States",
  "screening_score": 35
}

--- neighbor id=1036  similarity=1.00 ---
{
  "job_title": "data scientist",
  "experience": null,
  "education": null,
  "other_info": null,
  "location": "United State

In [16]:
print(best_alpha)

0.8


## 8. End-to-end demo on a brand-new candidate

Same walkthrough as the one worked through earlier, now runnable
directly against the built index.

In [17]:
new_candidate_row = {
    "job_title": "Data Analyst",
    "experience": "3 years",
    "education": "B.Sc. Statistics",
    "other_info": "SQL, Power BI, Python",
    "location": "US",
}
new_candidate = build_input_text(new_candidate_row) 
result = score_with_rag(new_candidate, alpha=best_alpha, k=K)
inspect_retrieval(new_candidate_row, result, None, train_df)

TARGET CANDIDATE
{
  "job_title": "Data Analyst",
  "experience": "3 years",
  "education": "B.Sc. Statistics",
  "other_info": "SQL, Power BI, Python",
  "location": "US"
}

predicted_score (final): 81.0

NEIGHBORS (from train_df)

--- neighbor id=605  similarity=0.93 ---
{
  "job_title": "data analyst",
  "experience": "2 years",
  "education": null,
  "other_info": "python, r, sql, tableau, power bi, cloud platforms skills",
  "location": "United States",
  "screening_score": 85
}

--- neighbor id=45  similarity=0.93 ---
{
  "job_title": "strategic data analyst",
  "experience": "9 years",
  "education": null,
  "other_info": "advanced analytics, machine learning, power bi, python, sql, aws, azure, expertise",
  "location": "United States",
  "screening_score": 95
}

--- neighbor id=254  similarity=0.93 ---
{
  "job_title": "data analyst",
  "experience": "5 years",
  "education": null,
  "other_info": "python, sql, tableau, ai skills",
  "location": "United States",
  "screening_sc

In [18]:
base = {k: (v.lower() if isinstance(v, str) else v) for k, v in new_candidate_row.items()}

def reg(row):
    return score_and_embed(build_input_text(row))[0]

print(f"{'full input':22s} {reg(base):.1f}")
for f in ["job_title", "experience", "education", "other_info", "location"]:
    print(f"{'without ' + f:22s} {reg({**base, f: None}):.1f}")
print(f"{'experience=student':22s} {reg({**base, 'experience': 'student/intern-level'}):.1f}")
print(f"{'location=UAE':22s} {reg({**base, 'location': 'united arab emirates'}):.1f}")

full input             79.5
without job_title      73.5
without experience     77.5
without education      79.0
without other_info     77.5
without location       62.5
experience=student     80.0
location=UAE           82.5


## 9. What's saved in `/kaggle/working`

- `candidate_faiss.index` — the FAISS index built from `train_df`.
- `candidate_faiss_metadata.csv` — `id` / `job_title` / `screening_score` per index row, aligned by position (`row`).

To score new candidates later without rerunning Sections 0–3: reload the
adapter (Section 1), `faiss.read_index("candidate_faiss.index")`, reload
the metadata CSV to rebuild `train_scores`/`train_titles`, and call
`score_with_rag` with the locked `best_alpha` from Section 5.